# 07 Results Freeze

Final reviewer evidence matrix and artifact freeze.

## Setup

Clone/pull latest code into `/content/ECG-RAMBA`, restore verified Drive mirror artifacts, and define a logging command helper.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import time

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print(f'Drive mount skipped or already mounted: {exc}')

DRIVE_ROOT = Path('/content/drive/MyDrive/ECG-Ramba')
MIRROR_REVISION_ROOT = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'
REPO_URL = 'https://github.com/BrianNguyen29/ECG-RAMBA.git'
BRANCH = os.environ.get('ECG_RAMBA_BRANCH', 'main')
REPO_DIR = Path('/content/ECG-RAMBA')

os.chdir('/content')
if REPO_DIR.exists() and not (REPO_DIR / '.git').exists():
    raise RuntimeError(f'{REPO_DIR} exists but is not a git checkout. Rename it or use a fresh runtime.')
if not REPO_DIR.exists():
    print(f'$ git clone --branch {BRANCH} {REPO_URL} "{REPO_DIR}"')
    subprocess.run(f'git clone --branch {BRANCH} {REPO_URL} "{REPO_DIR}"', shell=True, check=True)

os.chdir(REPO_DIR)

def run(cmd, check=True, log_path=None, tail_lines=120, cwd=None):
    run_cwd = Path(cwd) if cwd else Path.cwd()
    print(f'$ {cmd}', flush=True)
    if log_path is None:
        log_dir = run_cwd / 'reports' / 'revision' / 'logs'
        log_dir.mkdir(parents=True, exist_ok=True)
        log_path = log_dir / f'notebook_command_{time.strftime("%Y%m%d_%H%M%S")}.log'
    else:
        log_path = Path(log_path)
        if not log_path.is_absolute():
            log_path = run_cwd / log_path
        log_path.parent.mkdir(parents=True, exist_ok=True)

    with log_path.open('w', encoding='utf-8') as log:
        proc = subprocess.Popen(
            cmd,
            shell=True,
            cwd=str(run_cwd),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        rc = proc.wait()
    print(f'Command log: {log_path}')
    if check and rc:
        lines = log_path.read_text(encoding='utf-8', errors='replace').splitlines()
        for line in lines[-tail_lines:]:
            print(line)
        raise subprocess.CalledProcessError(rc, cmd)
    return subprocess.CompletedProcess(cmd, rc)

run('git fetch origin', check=False)
run(f'git checkout {BRANCH}', check=True)
run(f'git pull --ff-only --autostash origin {BRANCH}', check=True)

if MIRROR_REVISION_ROOT.exists():
    run(
        f'python -u scripts/revision/artifact_mirror.py restore --mirror-root "{MIRROR_REVISION_ROOT}" --replace-mismatched',
        log_path='reports/revision/logs/final_evidence_restore.log',
    )
else:
    print('Mirror not found; continuing with repo-local reports/revision:', MIRROR_REVISION_ROOT)

print('cwd       :', Path.cwd())
print('drive_root:', DRIVE_ROOT)
print('repo_dir  :', REPO_DIR)
print('branch    :', BRANCH)
run('git rev-parse --short HEAD', check=False)
run('git status --short --branch', check=False)


## Required Input Check

Validate the revision-plan CSV files and the final evidence inputs before building the final matrix. This fails early if paired ResNet or other required artifacts are missing from the restored mirror.


In [ ]:
import pandas as pd

revision_root = Path('reports/revision')
plan_csvs = [
    Path('docs/revision_plan/experiment_registry.csv'),
    Path('docs/revision_plan/claim_evidence_map.csv'),
    Path('docs/revision_plan/task_board.csv'),
]
for csv_path in plan_csvs:
    df = pd.read_csv(csv_path)
    print(f'{csv_path}: rows={len(df)} cols={len(df.columns)}')

required_final_inputs = {
    'calibration': revision_root / 'metrics' / 'calibration_ci_oof_final_ema_predictions.json',
    'pooling': revision_root / 'metrics' / 'pooling_sensitivity.csv',
    'baseline': revision_root / 'metrics' / 'baseline_summary.csv',
    'component': revision_root / 'metrics' / 'component_check_summary.json',
    'hrv_domain': revision_root / 'metrics' / 'hrv_domain_summary.csv',
    'robustness': revision_root / 'metrics' / 'robustness_summary.csv',
    'paired_minirocket': revision_root / 'metrics' / 'paired_full_vs_minirocket_comparison.json',
    'paired_resnet': revision_root / 'metrics' / 'paired_full_vs_resnet_comparison.json',
    'a0_status': revision_root / 'a0_resolution_status.json',
    'claim_map': Path('docs/revision_plan/claim_evidence_map.csv'),
    'task_board': Path('docs/revision_plan/task_board.csv'),
}
optional_final_inputs = {
    'paired_raw_mamba': revision_root / 'metrics' / 'paired_full_vs_raw_mamba_comparison.json',
}
def collect_missing_final_inputs(verbose=True):
    missing = []
    for label, path in required_final_inputs.items():
        exists = path.exists()
        if verbose:
            print(f'{label:18}: exists={exists} size={path.stat().st_size if exists else None} path={path}')
        if not exists:
            missing.append(f'{label}={path}')
    return missing

missing_inputs = collect_missing_final_inputs(verbose=True)
if missing_inputs:
    print('\nRequired inputs are missing from the active repo. Attempting verified Drive mirror restore before failing.')
    if 'MIRROR_REVISION_ROOT' not in globals():
        raise RuntimeError('MIRROR_REVISION_ROOT is undefined. Run the Setup cell first.')
    if not MIRROR_REVISION_ROOT.exists():
        raise FileNotFoundError(f'Mirror root does not exist: {MIRROR_REVISION_ROOT}')
    run(
        f'python -u scripts/revision/artifact_mirror.py restore --mirror-root "{MIRROR_REVISION_ROOT}" --replace-mismatched',
        log_path='reports/revision/logs/final_evidence_required_restore.log',
    )
    print('\nRechecking required final evidence inputs after mirror restore:')
    missing_inputs = collect_missing_final_inputs(verbose=True)
if missing_inputs:
    raise FileNotFoundError('Missing required final evidence inputs after mirror restore: ' + '; '.join(missing_inputs))

print('Optional final evidence inputs:')
for label, path in optional_final_inputs.items():
    exists = path.exists()
    print(f'{label:18}: exists={exists} size={path.stat().st_size if exists else None} path={path}')

paired_resnet = json.loads(required_final_inputs['paired_resnet'].read_text(encoding='utf-8'))
resnet_metrics = paired_resnet.get('metrics', {})
print('Paired ResNet interpretations:')
for metric in ['pr_auc_macro', 'roc_auc_macro', 'f1_macro', 'brier_macro', 'ece_macro']:
    row = resnet_metrics.get(metric, {})
    print(f"  {metric}: {row.get('interpretation')} full={row.get('full_value')} comparator={row.get('comparator_value')}")

paired_raw_path = optional_final_inputs['paired_raw_mamba']
if paired_raw_path.exists():
    paired_raw = json.loads(paired_raw_path.read_text(encoding='utf-8'))
    raw_metrics = paired_raw.get('metrics', {})
    print('Paired Raw Mamba interpretations:')
    for metric in ['pr_auc_macro', 'roc_auc_macro', 'f1_macro', 'brier_macro', 'ece_macro']:
        row = raw_metrics.get(metric, {})
        print(f"  {metric}: {row.get('interpretation')} full={row.get('full_value')} comparator={row.get('comparator_value')}")
else:
    print('Paired Raw Mamba optional input is absent; final matrix will keep Raw Mamba incomplete until Notebook 04 produces it.')


## Final Evidence Matrix

Build claim-level evidence, blocker status, robustness claim rows, and reviewer-safe wording from frozen artifacts.

In [ ]:
import pandas as pd

run(
    'python -u scripts/revision/13_final_evidence_matrix.py --strict',
    log_path='reports/revision/logs/final_evidence_matrix.log',
)

matrix = pd.read_csv('reports/revision/tables/table_final_evidence_matrix.csv')
safe = pd.read_csv('reports/revision/tables/table_final_safe_wording.csv')
robust = pd.read_csv('reports/revision/tables/table_final_robustness_claims.csv')
blockers = pd.read_csv('reports/revision/tables/table_final_blocker_status.csv')

print('Evidence matrix:')
display(matrix[['claim_id', 'claim_topic', 'evidence_status', 'key_numbers', 'blocker']])
print('Safe wording:')
display(safe[['claim_id', 'evidence_status', 'safe_wording']])
print('Robustness claim rows:', len(robust))
display(robust[['stress_test', 'metric', 'degradation_interpretation', 'stressed_performance_interpretation']])
print('A0 blocker status:')
display(blockers[['blocker_id', 'resolution_status', 'restriction']])


## Validation Gate

Fail if the final evidence matrix is incomplete or if blocked/limited claims are accidentally marked as fully supported.

In [ ]:
payload = json.loads(Path('reports/revision/metrics/final_evidence_matrix.json').read_text(encoding='utf-8'))
manifest = json.loads(Path('reports/revision/manifests/final_evidence_matrix_manifest.json').read_text(encoding='utf-8'))
required_outputs = [
    Path('reports/revision/metrics/final_evidence_matrix.json'),
    Path('reports/revision/tables/table_final_evidence_matrix.csv'),
    Path('reports/revision/tables/table_final_safe_wording.csv'),
    Path('reports/revision/tables/table_final_blocker_status.csv'),
    Path('reports/revision/tables/table_final_robustness_claims.csv'),
    Path('reports/revision/manifests/final_evidence_matrix_manifest.json'),
]
missing = [str(path) for path in required_outputs if not path.exists()]
if missing:
    raise FileNotFoundError('Missing final evidence outputs: ' + '; '.join(missing))
if payload.get('final_ready_for_rebuttal') is not True:
    raise RuntimeError('Final evidence matrix is not ready for rebuttal use. Inspect unresolved_blockers/contract_issues.')
if payload.get('all_claims_supported') is True:
    raise RuntimeError('Unexpected all_claims_supported=True; blocked/limited claims must remain explicit.')
if len(payload.get('evidence_matrix', [])) != 6:
    raise RuntimeError(f"Final evidence matrix should have 6 claim rows, found {len(payload.get('evidence_matrix', []))}")
if len(payload.get('robustness_claims', [])) != 30:
    raise RuntimeError(f"Final robustness claim table should have 30 rows, found {len(payload.get('robustness_claims', []))}")
if payload.get('missing_inputs'):
    raise RuntimeError('Final evidence matrix reports missing inputs: ' + '; '.join(payload.get('missing_inputs', [])))
if manifest.get('final_ready_for_rebuttal') is not True:
    raise RuntimeError('Final evidence manifest does not mark rebuttal readiness.')
matrix_by_claim = {row.get('claim_id'): row for row in payload.get('evidence_matrix', [])}
c01 = matrix_by_claim.get('C01', {})
c02 = matrix_by_claim.get('C02', {})
c03 = matrix_by_claim.get('C03', {})
c04 = matrix_by_claim.get('C04', {})
if c01.get('source_claim_status') == 'blocked_fair_baselines_missing':
    raise RuntimeError('C01 source_claim_status is stale after Raw Mamba completion.')
for stale_phrase in ['external-transfer, or efficiency evidence', 'If domain AUC is high', 'Robustness rows=']:
    blob = '\n'.join(str(row) for row in payload.get('evidence_matrix', []))
    if stale_phrase in blob:
        raise RuntimeError(f'Stale final wording remains: {stale_phrase}')
if 'ResNet1D/CNN is stronger on PR-AUC, ROC-AUC, F1, Brier, and ECE' not in c02.get('safe_wording', ''):
    raise RuntimeError('C02 safe wording must state the ResNet comparator-specific result.')
if 'near-perfect HRV domain classifier' not in c03.get('safe_wording', ''):
    raise RuntimeError('C03 safe wording must state observed HRV domain sensitivity.')
if 'representation separation remains unproven' not in c04.get('key_numbers', ''):
    raise RuntimeError('C04 key_numbers must not use robustness counts as representation evidence.')
print('Final evidence matrix validated.')
print('All claims supported:', payload.get('all_claims_supported'))
print('Contract issues:', payload.get('contract_issues'))


## Inventory And Stable Mirror

Refresh the artifact manifest and publish final evidence outputs back to Drive.

In [ ]:
run(
    'python -u scripts/revision/05_artifact_inventory.py',
    log_path='reports/revision/logs/final_artifact_inventory.log',
)
run(
    f'python -u scripts/revision/artifact_mirror.py publish --mirror-root "{MIRROR_REVISION_ROOT}"',
    log_path='reports/revision/logs/final_evidence_mirror_publish.log',
)


## Convenience Drive Copies

Copy final reviewer tables to a short Drive path for manual download/opening. These are convenience copies; the canonical artifacts remain under `reports/revision/` and the verified mirror.

In [ ]:
import shutil

FINAL_TABLE_EXPORT_DIR = DRIVE_ROOT / 'final_evidence_tables'
FINAL_TABLE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
final_table_sources = [
    Path('reports/revision/metrics/final_evidence_matrix.json'),
    Path('reports/revision/tables/table_final_evidence_matrix.csv'),
    Path('reports/revision/tables/table_final_safe_wording.csv'),
    Path('reports/revision/tables/table_final_blocker_status.csv'),
    Path('reports/revision/tables/table_final_robustness_claims.csv'),
    Path('reports/revision/manifests/final_evidence_matrix_manifest.json'),
]
for source in final_table_sources:
    if not source.exists():
        raise FileNotFoundError(source)
    destination = FINAL_TABLE_EXPORT_DIR / source.name
    shutil.copy2(source, destination)
    print(f'Copied: {destination} size={destination.stat().st_size}')

readme = FINAL_TABLE_EXPORT_DIR / 'README.md'
readme.write_text(
    '# ECG-RAMBA final evidence tables\n\n'
    'Convenience copies generated by notebooks/07_results_freeze.ipynb.\n\n'
    '- table_final_evidence_matrix.csv: claim-level evidence status and key numbers.\n'
    '- table_final_safe_wording.csv: reviewer/manuscript-safe wording.\n'
    '- table_final_blocker_status.csv: A0 blocker decisions and restrictions.\n'
    '- table_final_robustness_claims.csv: stress/metric-specific robustness CI interpretations.\n'
    '- final_evidence_matrix.json: full machine-readable payload.\n\n'
    'Canonical verified artifacts remain under reports/revision and the revision_artifacts mirror.\n',
    encoding='utf-8',
)
print('Convenience export dir:', FINAL_TABLE_EXPORT_DIR)


## Output Summary

In [ ]:
expected = [
    'reports/revision/metrics/final_evidence_matrix.json',
    'reports/revision/tables/table_final_evidence_matrix.csv',
    'reports/revision/tables/table_final_safe_wording.csv',
    'reports/revision/tables/table_final_blocker_status.csv',
    'reports/revision/tables/table_final_robustness_claims.csv',
    'reports/revision/manifests/final_evidence_matrix_manifest.json',
    'reports/revision/manifests/artifacts_manifest.json',
    'reports/revision/manifests/artifacts_manifest.csv',
    str(DRIVE_ROOT / 'final_evidence_tables' / 'table_final_evidence_matrix.csv'),
    str(DRIVE_ROOT / 'final_evidence_tables' / 'table_final_safe_wording.csv'),
]
for item in expected:
    path = Path(item)
    print(f'{item}: exists={path.exists()} size={path.stat().st_size if path.exists() else None}')

optional_expected = [
    'reports/revision/metrics/paired_full_vs_raw_mamba_comparison.json',
    'reports/revision/tables/table_paired_full_vs_raw_mamba.csv',
    'reports/revision/manifests/paired_full_vs_raw_mamba_manifest.json',
]
print('\nOptional Raw Mamba evidence artifacts:')
for item in optional_expected:
    path = Path(item)
    print(f'{item}: exists={path.exists()} size={path.stat().st_size if path.exists() else None}')

payload = json.loads(Path('reports/revision/metrics/final_evidence_matrix.json').read_text(encoding='utf-8'))
print('\nFinal guidance:')
for key, value in payload.get('claim_guidance', {}).items():
    print(f'- {key}: {value}')
print('\nConvenience Drive folder:', DRIVE_ROOT / 'final_evidence_tables')
print('Notebook 07 complete. Use table_final_evidence_matrix.csv and table_final_safe_wording.csv as the manuscript/rebuttal source of truth.')
